# 2. Defender XDR Incident Response Procedures

The SC-200 exam asks you to respond to incidents from EACH Defender product. This notebook covers the response procedures for each.

## Defender for Office 365 — Email incidents

### When you get a phishing alert

| Step | Action | Portal location |
|------|--------|-----------------|
| 1 | Check email headers and sender | Email entity page → Header analysis |
| 2 | Check URLs/attachments | Email entity → URL/attachment detonation results |
| 3 | Find all recipients | Advanced Hunting: `EmailEvents \| where SenderFromAddress == "..."` |
| 4 | Soft-delete from all mailboxes | Actions → Soft delete email |
| 5 | Block sender/domain | Tenant Allow/Block List |
| 6 | Check if any user clicked | `UrlClickEvents \| where Url contains "..."` |
| 7 | If clicked: investigate that user | User entity page → expand investigation |

### Automatic attack disruption

Defender XDR can **automatically** disrupt attacks:
- Disable compromised accounts
- Block malicious OAuth apps
- Contain compromised devices

This runs BEFORE a human analyst even sees the alert.

## Defender for Endpoint — Device incidents

### When you get a malware/exploitation alert

| Step | Action | How |
|------|--------|-----|
| 1 | Review device timeline | Device page → Timeline (shows process tree) |
| 2 | Check alert process tree | Alert → Process tree (parent → child chain) |
| 3 | Isolate the device | Device page → Actions → Isolate (network isolation) |
| 4 | Collect investigation package | Actions → Collect investigation package (forensic data) |
| 5 | Run AV scan | Actions → Run antivirus scan |
| 6 | Live response (if needed) | Actions → Initiate live response session |
| 7 | Release from isolation | Actions → Release from isolation |

### Live response commands

```
# Connect to device
connections                    # Show network connections
processes                     # List running processes
registry list HKLM\...       # Read registry
getfile C:\Users\...\mal.exe  # Download file for analysis
remediate file C:\...\mal.exe # Quarantine a file
run script.ps1               # Run forensic script
```

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

# Simulate the device timeline investigation
print('=== Device Timeline: laptop-alice ===\n')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'DeviceEvents',
    'filter': {'DeviceName': 'laptop-alice'},
    'limit': 20,
})
events = r.json()['results']

suspicious_tools = {'mimikatz.exe', 'psexec.exe', 'cmd.exe', 'powershell.exe', 'certutil.exe', 'curl'}

for e in events:
    is_suspicious = e['FileName'] in suspicious_tools
    icon = '🔴' if is_suspicious else '⬜'
    print(f'{icon} [{e["timestamp"][:19]}] {e["FileName"]} ({e["ActionType"]})')
    print(f'   Path: {e["FolderPath"]}  User: {e["AccountName"]}')
    if is_suspicious:
        print(f'   ⚠️  SUSPICIOUS: known attack tool!')
    print()

print('💡 In real Defender for Endpoint, the timeline includes:')
print('   - Process creation trees (parent → child)')
print('   - File modifications')
print('   - Registry changes')
print('   - Network connections')
print('   - Script content (PowerShell, WMI)')

## Defender for Identity — Identity incidents

Detects on-premises AD attacks:

| Detection | What it catches |
|-----------|----------------|
| Pass-the-hash | Stolen NTLM hash used for authentication |
| Pass-the-ticket | Stolen Kerberos ticket replayed |
| Golden ticket | Forged Kerberos TGT using KRBTGT hash |
| DCSync | Replication request to extract password hashes |
| Reconnaissance | LDAP, DNS, SMB enumeration |
| Lateral movement | Remote execution via PsExec, WMI, etc. |

## Defender for Cloud Apps — SaaS incidents

| Alert type | Response |
|-----------|----------|
| Suspicious OAuth app | Revoke app consent, ban the app |
| Mass download/delete | Suspend user, investigate timeline |
| Impossible travel | Verify with user, require MFA |
| Shadow IT detection | Review app risk score, block or sanction |

## Microsoft Purview — Compliance incidents

| Alert type | Response |
|-----------|----------|
| DLP policy match | Review the shared content, educate user |
| Insider risk alert | Open investigation, correlate with HR data |
| eDiscovery alert | Coordinate with legal, preserve content |

## Incident classification cheat sheet

| Classification | When to use | Example |
|---------------|-------------|----------|
| **True Positive** | Real attack, action taken | Confirmed phishing + credential theft |
| **Benign Positive** | Real activity, not malicious | Authorized pen test triggered alert |
| **False Positive** | Wrong detection | Legit app flagged as malware |
| **Undetermined** | Insufficient evidence | Suspicious but can't confirm |

### Exam tip

- Always **classify** incidents when closing — this trains the ML models.
- **False positives** should lead to rule tuning (suppress or modify the detection).
- **Automatic attack disruption** can disable users and isolate devices BEFORE you investigate.
- Know the difference between **soft delete** (recoverable) and **hard delete** (permanent) for email remediation.